<a href="https://colab.research.google.com/github/Ankan2309/Ankan2309/blob/main/Bike_Sharing_Demand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

import warnings

warnings.filterwarnings('ignore')

In [62]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [63]:
import zipfile

# Unzip the dataset if it hasn't been already
# Assuming train.csv and test.csv are directly inside the zip file
with zipfile.ZipFile('/content/bike-sharing-demand.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

train = pd.read_csv("/content/train.csv")
test = pd.read_csv("/content/test.csv")

print(train.shape)
print(test.shape)

train.head()

(10886, 12)
(6493, 9)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [64]:
train_len = len(train)
df = pd.concat([train, test], axis=0).reset_index(drop=True)

In [65]:
df.shape

(17379, 12)

In [66]:
df.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3.0,13.0,16.0
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8.0,32.0,40.0
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5.0,27.0,32.0
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3.0,10.0,13.0
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0.0,1.0,1.0


In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    17379 non-null  object 
 1   season      17379 non-null  int64  
 2   holiday     17379 non-null  int64  
 3   workingday  17379 non-null  int64  
 4   weather     17379 non-null  int64  
 5   temp        17379 non-null  float64
 6   atemp       17379 non-null  float64
 7   humidity    17379 non-null  int64  
 8   windspeed   17379 non-null  float64
 9   casual      10886 non-null  float64
 10  registered  10886 non-null  float64
 11  count       10886 non-null  float64
dtypes: float64(6), int64(5), object(1)
memory usage: 1.6+ MB


In [68]:
df.describe()

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
count,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,17379.000000,10886.000000,10886.000000,10886.000000
mean,2.501640,0.028770,0.682721,1.425283,20.376474,23.788755,62.722884,12.736540,36.021955,155.552177,191.574132
std,1.106918,0.167165,0.465431,0.639357,7.894801,8.592511,19.292983,8.196795,49.960477,151.039033,181.144454
min,1.000000,0.000000,0.000000,1.000000,0.820000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,2.000000,0.000000,0.000000,1.000000,13.940000,16.665000,48.000000,7.001500,4.000000,36.000000,42.000000
50%,3.000000,0.000000,1.000000,1.000000,20.500000,24.240000,63.000000,12.998000,17.000000,118.000000,145.000000
75%,3.000000,0.000000,1.000000,2.000000,27.060000,31.060000,78.000000,16.997900,49.000000,222.000000,284.000000
max,4.000000,1.000000,1.000000,4.000000,41.000000,50.000000,100.000000,56.996900,367.000000,886.000000,977.000000


In [69]:
df["datetime"] = pd.to_datetime(df["datetime"])

In [70]:
df["hour"] = df["datetime"].dt.hour

df["day"] = df["datetime"].dt.day

df["month"] = df["datetime"].dt.month

df['year'] = df['datetime'].dt.year

df['dayofweek'] = df['datetime'].dt.dayofweek


In [71]:
df['season_label'] = df['season'].map({1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"})

In [72]:
df['weather_label'] = df['weather'].map({1: "Clear", 2: "Cloudy", 3: "Rainy", 4: "Heavy_Rain"})

In [73]:
df['day_name'] = df['dayofweek'].map({0:"Mon", 1:"Tue", 2:"Wed", 3:"Thu", 4:"Fri", 5:"Sat", 6:"Sun"})

In [74]:
df['is_peak_hour'] = 0
df.loc[(df['workingday'] == 1) & (df['hour'].isin([8, 17, 18])), 'is_peak_hour'] = 1

In [75]:
df = pd.get_dummies(df, columns=["season", "weather"], drop_first=False)

Train/Test Spilt

In [76]:
train_final = df.iloc[:train_len].copy()
test_final = df.iloc[train_len:].copy()

In [77]:
y_train = np.log1p(train["count"])

In [78]:
drop_cols = [
    "datetime",
    "casual",
    "registered",
    "count",
    "season_label",
    "weather_label",
    "day_name"
]

x_train = train_final.drop(columns=drop_cols, errors="ignore")
x_test = test_final.drop(columns=drop_cols, errors="ignore")

Model Training

In [79]:
from sklearn.ensemble import RandomForestRegressor

In [80]:
model = RandomForestRegressor(n_estimators=500, max_depth=15, random_state=42, n_jobs=-1)

In [81]:
model.fit(x_train, y_train)

RandomForestRegressor(max_depth=15, n_estimators=500, n_jobs=-1,
                      random_state=42)

In [82]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

x_tr, x_val, y_tr, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42
)

model.fit(x_tr, y_tr)

val_pred = model.predict(x_val)

val_pred_real = np.expm1(val_pred)
y_val_real = np.expm1(y_val)

r2 = r2_score(y_val_real, val_pred_real)
rmse = np.sqrt(mean_squared_error(y_val_real, val_pred_real))
mae = mean_absolute_error(y_val_real, val_pred_real)

print("R2 Score:", r2)
print("RMSE:", rmse)
print("MAE:", mae)

R2 Score: 0.9478103131675093
RMSE: 41.50447742227773
MAE: 25.56747340908193


Prediction

In [83]:
pred_log = model.predict(x_test)

pred = np.expm1(pred_log)
pred = np.maximum(pred, 0)

Submission

In [84]:
submission = pd.DataFrame({"datetime": test["datetime"],"count": pred})

submission.head()

,datetime,count
0,2011-01-20 00:00:00,13.741402
1,2011-01-20 01:00:00,5.754206
2,2011-01-20 02:00:00,4.551619
3,2011-01-20 03:00:00,3.277570
4,2011-01-20 04:00:00,2.511211


In [85]:
submission.to_csv("submission.csv", index=False)

In [86]:
print("submission.csv created successfully")

submission.csv created successfully


In [87]:
print("submission.csv created successfully")

submission.csv created successfully
